In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient
from langchain_community.utilities import SQLDatabase

tavily_client = TavilyClient()

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")


@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [3]:
from dataclasses import dataclass

@dataclass
class UserRole:
    user_role: str = "external"

In [4]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

@wrap_model_call
def dynamic_tool_call(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:

    """Dynamically call tools based on the runtime context"""

    user_role = request.runtime.context.user_role
    
    if user_role == "internal":
        pass # internal users get access to all tools
    else:
        tools = [web_search] # external users only get access to web search
        request = request.override(tools=tools) 

    return handler(request)

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search, sql_query],
    middleware=[dynamic_tool_call],
    context_schema=UserRole
)

In [6]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "external"}
)

print(response["messages"][-1].content)

I don’t have access to your database, so I can’t give a number yet. Please tell me which database you’re using and the table/collection name (is it artists?). If you want, you can also paste a snippet of the schema.

In the meantime, here are quick queries for common setups:

- SQL databases (PostgreSQL, MySQL, SQLite, SQL Server)
  - Total artists: SELECT COUNT(*) AS total_artists FROM artists;
  - Distinct by id (if id isn’t the primary key): SELECT COUNT(DISTINCT id) AS total_artists FROM artists;
  - Only active artists (if you have an active flag): SELECT COUNT(*) AS total_artists FROM artists WHERE active = true;

- MongoDB
  - Total documents: db.artists.countDocuments({});
  - If you only want distinct IDs: db.artists.distinct("_id").length

- REST API (if you have an endpoint)
  - GET /api/artists/count (or GET /api/artists?limit=0 or /artists?status=active depending on API)

If you share the database type and any filters (e.g., only active artists), I can give you the exact c

In [7]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How many artists are in the database?")]},
    context={"user_role": "internal"}
)

print(response["messages"][-1].content)

There are 275 artists in the database.
